In [0]:
-- =======================================
-- GOLD: Materialized views on SILVER
-- =======================================
CREATE OR REPLACE TABLE sunny_bay_roastery.gold.d_date AS
SELECT * FROM sunny_bay_roastery.silver.s_date;
CREATE OR REPLACE TABLE sunny_bay_roastery.gold.d_store AS
SELECT * FROM sunny_bay_roastery.silver.s_store;
CREATE OR REPLACE TABLE sunny_bay_roastery.gold.d_customer AS
SELECT * FROM sunny_bay_roastery.silver.s_customer;
CREATE OR REPLACE TABLE sunny_bay_roastery.gold.d_product AS
SELECT * FROM sunny_bay_roastery.silver.s_product;
CREATE OR REPLACE TABLE   sunny_bay_roastery.gold.f_coffee_sales AS
SELECT
    fcs.*,
    dp.list_price_usd * fcs.quantity_sold                       AS gross_revenue_usd,
    (dp.list_price_usd * fcs.quantity_sold) / (1 + ds.tax_rate) AS net_revenue_usd,
    ds.tax_rate * dp.list_price_usd * fcs.quantity_sold  AS vat_usd,
    dp.cost_of_goods_usd * fcs.quantity_sold AS cost_of_goods_usd,
    (dp.list_price_usd * fcs.quantity_sold) * 1.1 AS gross_revenue_eur

FROM sunny_bay_roastery.silver.s_sales fcs
JOIN sunny_bay_roastery.silver.s_product dp
  ON fcs.product_key = dp.product_key
JOIN sunny_bay_roastery.silver.s_store ds
  ON fcs.store_key = ds.store_key;

CREATE OR REPLACE TABLE sunny_bay_roastery.gold.total_revenue_by_year AS
SELECT
    store_key AS store_key,
    SUM(gross_revenue_usd) AS total_gross_revenue_usd
FROM sunny_bay_roastery.gold.f_coffee_sales
GROUP BY store_key;
